# Pretrained Networks

Training a large CNN takes millions of labelled images and days of computing time. You rarely have either. Research groups have trained networks on **ImageNet** and published the learnt weights, so you can download a network that already recognises 1,000 kinds of objects and use it directly. In this notebook you look inside two such networks, VGG16 and MobileNetV2, classify photos with MobileNetV2, and find out where a pretrained network reaches its limits.

## Learning Objectives

By the end of this notebook, you should be able to:

- Load a pretrained network from `keras.applications` and classify a photo with it.
- Prepare a photo in the format a pretrained network expects: its size, a batch axis and the network's preprocessing.
- Calculate the parameters of standard and depthwise separable convolutions, and identify where most parameters of VGG16 and MobileNetV2 are.
- Plot the feature maps of an early and a late layer of a pretrained network.
- Identify when a pretrained network cannot classify an image correctly.

## Setup

In [ ]:
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
from keras.applications import mobilenet_v2

PHOTOS = Path("../assets") / "photos"  # example photos from the Caltech-101 dataset and one bean leaf

## 1. Pretrained Networks

A **pretrained network** is a network that someone else has already trained on a large dataset, published together with its learnt weights. You can use it in two ways:

- **directly**, to classify your own photos, as long as they show the kind of objects the network was trained on (this notebook);
- as the starting point for a new task with **transfer learning** (notebook 02).

### Why Use Them

- You skip the long and expensive training: the weights are a download of a few megabytes.
- They reach a high accuracy that a small team with a small dataset could not reach on its own.
- Their convolutional layers have learnt general features, such as edges, textures and shapes, that are useful for almost any photo.

### Some Well-Known Networks

[`keras.applications`](https://keras.io/api/applications/) provides many of these networks with their weights. Three of them:

| Network | Year | Parameters | Weights to download | Idea |
|---|---|---|---|---|
| VGG16 | 2014 | 138 million | 528 MB | 13 convolutional layers with 3 × 3 filters, then three large dense layers |
| ResNet50 | 2015 | 26 million | 98 MB | 50 layers; shortcuts let the signal skip layers, so very deep networks can be trained |
| MobileNetV2 | 2018 | 3.5 million | 14 MB | depthwise separable convolutions: much less computation, designed for phones (section 3) |

The weights are downloaded the first time you create a network with `weights="imagenet"` and are stored in `~/.keras/models`, so later runs load them from disk.

## 2. VGG16 and ImageNet

### The ImageNet Dataset

[ImageNet](https://www.image-net.org/) is a collection of millions of labelled photos. Its yearly competition (2010 to 2017) used a subset of about 1.2 million training images in **1,000 classes**, from "goldfish" and "llama" to "laptop" and "umbrella". The competition measured the **top-5 error rate**: the share of images whose correct class is not among the five most probable classes. In 2011, before the first CNN entered, the best top-5 error rate was about 26 %; by 2015 CNNs had brought it below 4 %, and the networks that won became standard tools.

### Inside VGG16

[VGG16](https://keras.io/api/applications/vgg/) is the classic example of a deep CNN. It is built from the parts you know: convolutional layers with 3 × 3 filters (Keras and many texts call a filter a **kernel**, as in the argument `kernel_size`) and `"same"` padding, grouped into five blocks that each end with 2 × 2 max pooling, followed by `Flatten` and three dense layers.

<img src="../assets/vgg16.png" width="1000">

With `weights=None`, Keras builds the architecture without downloading the weights, which is enough to inspect it.

**Your turn:** before you run the next cell, calculate the parameters of the first convolutional layer (64 filters of 3 × 3 on a colour image) and of the first dense layer (4,096 neurons on the 7 × 7 × 512 output of the last block), with the formulas from the convolutional networks topic.

<details>
<summary>Show the answer</summary>

- First convolutional layer: $(3 \cdot 3 \cdot 3 + 1) \cdot 64 = 1{,}792$.
- First dense layer: `Flatten` turns 7 × 7 × 512 into 25,088 values, so $25{,}088 \cdot 4{,}096 + 4{,}096 = 102{,}764{,}544$.

</details>

In [ ]:
vgg16 = keras.applications.VGG16(weights=None)  # the architecture only: nothing is downloaded
vgg16.summary()

In [ ]:
first_dense = vgg16.get_layer("fc1").count_params()
print(f"First dense layer: {first_dense:,} of {vgg16.count_params():,} parameters "
      f"({first_dense / vgg16.count_params():.0%})")

The first dense layer holds 102,764,544 of the 138,357,544 parameters, 74 %. The three dense layers together hold 89 %, while the 13 convolutional layers, which do the actual work of finding features, need only 14,714,688 (11 %). This is the reason why later networks such as MobileNetV2 replace `Flatten` and the large dense layers with global average pooling: their convolutional base is just as capable, and the network is many times smaller.

## 3. Inside MobileNetV2

VGG16 is accurate but large: 138 million parameters and a 528 MB download are too much for a phone. [MobileNetV2](https://keras.io/api/applications/mobilenet/) reaches a similar accuracy with 3.5 million parameters, using two ideas.

**Depthwise separable convolutions.** MobileNetV2 replaces most standard convolutions with a cheaper pair of layers: a [**depthwise convolution**](https://keras.io/api/layers/convolution_layers/depthwise_convolution2d/) filters each channel separately with its own 3 × 3 filter, and a **pointwise convolution** (an ordinary `Conv2D` with 1 × 1 filters) then mixes the channels at each pixel. Panel A below compares the number of weights for 96 channels in and out. Eli Bendersky's article [Depthwise separable convolutions for machine learning](https://eli.thegreenplace.net/2018/depthwise-separable-convolutions-for-machine-learning/) explains both steps in detail, with a diagram for each.

**Inverted residual blocks.** MobileNetV2 is built from 17 blocks like the one in panel B: a 1 × 1 convolution expands the channels, a depthwise convolution filters them, and a second 1 × 1 convolution reduces them again. A **shortcut** adds the input of the block to its output, so the signal and the gradients can pass directly through the deep network.

<img src="../assets/mobilenet_blocks.png" width="900">

**Your turn:** before you run the next cell, calculate the number of weights and biases of a standard 3 × 3 convolution from 96 to 96 channels, and of a depthwise 3 × 3 convolution followed by a pointwise convolution with the same input and output. Each convolution here also has one bias per output channel.

<details>
<summary>Show the answer</summary>

- Standard: $(3 \cdot 3 \cdot 96 + 1) \cdot 96 = 83{,}040$.
- Depthwise: $(3 \cdot 3 + 1) \cdot 96 = 960$; pointwise: $(1 \cdot 1 \cdot 96 + 1) \cdot 96 = 9{,}312$; together $10{,}272$, about 8 times fewer.

</details>

In [ ]:
inputs = keras.Input(shape=(56, 56, 96))
standard = keras.layers.Conv2D(filters=96, kernel_size=3, padding="same")
depthwise = keras.layers.DepthwiseConv2D(kernel_size=3, padding="same")
pointwise = keras.layers.Conv2D(filters=96, kernel_size=1)
standard(inputs), pointwise(depthwise(inputs))  # calling the layers once creates their weights

print(f"Standard convolution:            {standard.count_params():,} parameters")
print(f"Depthwise + pointwise convolution: {depthwise.count_params() + pointwise.count_params():,} parameters")

The whole network stacks these blocks in seven stages. Like VGG16, it halves the height and the width several times while the number of channels grows, but it uses strides instead of pooling layers and ends with global average pooling and a single dense layer:

<img src="../assets/mobilenetv2.png" width="1000">

`keras.applications.MobileNetV2(weights=None)` builds it without weights, as before. The cell below counts where its parameters are.

In [ ]:
mobilenet_architecture = keras.applications.MobileNetV2(weights=None)
dense = mobilenet_architecture.get_layer("predictions").count_params()
total = mobilenet_architecture.count_params()
print(f"MobileNetV2: {total:,} parameters, {dense:,} of them in the dense layer ({dense / total:.0%})")

The two layers confirm the calculation: 83,040 parameters for the standard convolution against 10,272 for the depthwise separable one, about 8 times fewer for the same input and output shape. MobileNetV2 as a whole has 3,538,984 parameters, 39 times fewer than VGG16. Its single dense layer still holds 1,281,000 of them (36 %), because it connects 1,280 features to 1,000 classes; the convolutional base needs only about 2.3 million.

## 4. Classify a Photo with MobileNetV2

MobileNetV2 is small enough to download in seconds, so you use it to classify photos. Every network in `keras.applications` comes with two functions in its module:

- [`preprocess_input`](https://keras.io/api/applications/mobilenet/#preprocessinput-function) scales the pixel values the way the network was trained. For MobileNetV2 it maps 0 to 255 onto -1 to 1; VGG16 instead reorders the colour channels from RGB to BGR and subtracts the mean colour of ImageNet. A network fed differently scaled images gives poor predictions.
- [`decode_predictions`](https://keras.io/api/applications/mobilenet/#decodepredictions-function) turns the 1,000 output probabilities into readable class names.

The network also expects images of 224 × 224 pixels and, like every Keras model, a **batch**: an array of shape `(images, height, width, channels)`, even for a single image. The cells below go through these steps one at a time with a photo of a llama.

### Load the Model and the Photo

[`keras.utils.load_img`](https://keras.io/api/data_loading/image/#loadimg-function) loads the photo and resizes it to 224 × 224 pixels on the way.

In [ ]:
model = mobilenet_v2.MobileNetV2(weights="imagenet")  # downloads about 14 MB the first time

photo = keras.utils.load_img(PHOTOS / "llama.jpg", target_size=(224, 224))
print("Photo size:", photo.size)
photo

### Turn the Photo into a Batch

[`img_to_array`](https://keras.io/api/data_loading/image/#imgtoarray-function) turns the photo into a NumPy array of shape (height, width, channels). Indexing with `np.newaxis` adds the batch axis in front.

In [ ]:
pixels = keras.utils.img_to_array(photo)  # shape (224, 224, 3), values from 0 to 255
batch = pixels[np.newaxis]  # shape (1, 224, 224, 3): a batch of one image
print("Array shape:", pixels.shape, " batch shape:", batch.shape, " values from", batch.min(), "to", batch.max())

### Preprocess the Batch

In [ ]:
prepared = mobilenet_v2.preprocess_input(batch.copy())  # copy: preprocess_input changes the array it gets
print("After preprocess_input: values from", prepared.min(), "to", prepared.max())

### Predict and Decode

`predict` returns one row of 1,000 probabilities per image; `decode_predictions` turns the highest ones into class names.

In [ ]:
probabilities = model.predict(prepared, verbose=0)  # shape (1, 1000)
print("Output shape:", probabilities.shape)
for _, name, probability in mobilenet_v2.decode_predictions(probabilities, top=3)[0]:
    print(f"{name:20s} {probability:.3f}")

To see why the preprocessing matters, the cell below passes the same photo without `preprocess_input`, with pixel values from 0 to 255.

In [ ]:
raw_probabilities = model.predict(batch, verbose=0)
for _, name, probability in mobilenet_v2.decode_predictions(raw_probabilities, top=3)[0]:
    print(f"{name:20s} {probability:.3f}")

With the correct preprocessing, MobileNetV2 assigns a probability of 0.906 to the class llama; the next guesses, Arabian camel and ram, are related animals. Without preprocessing, the same network predicts a shower curtain with probability 0.738. The pixel values from 0 to 255 are far outside the range from -1 to 1 the network was trained on, so its filters respond to the wrong patterns. Forgetting `preprocess_input`, or using the function of another network, is one of the most common mistakes with pretrained models: the code runs without an error, but the predictions are wrong.

## 5. Classify a Set of Photos

The function below combines the steps, so you can classify all five example photos at once.

In [ ]:
def classify(model, module, path, size=224):
    """Return the three most probable ImageNet classes of a photo for a model from keras.applications."""
    pixels = keras.utils.img_to_array(keras.utils.load_img(path, target_size=(size, size)))
    probabilities = model.predict(module.preprocess_input(pixels[np.newaxis]), verbose=0)
    return [(name, probability) for _, name, probability in module.decode_predictions(probabilities, top=3)[0]]


photos = ["grand_piano.jpg", "laptop.jpg", "umbrella.jpg", "llama.jpg", "electric_guitar.jpg"]
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, name in zip(axes, photos):
    top3 = classify(model, mobilenet_v2, PHOTOS / name)
    ax.imshow(keras.utils.load_img(PHOTOS / name))
    ax.set_title("\n".join(f"{label} {p:.2f}" for label, p in top3), fontsize=9)
    ax.axis("off")
plt.show()

MobileNetV2 recognises all five objects. The photos come from Caltech-101, not from ImageNet. The grand piano (0.97), the llama (0.91) and the umbrella (0.89) get high probabilities. For the laptop the top class is "notebook" (0.63), with "laptop" second (0.14): ImageNet has both classes for the same kind of object, so the network divides its probability between them. The electric guitar is right but less certain (0.44), with banjo and acoustic guitar as the next guesses: guitar-like instruments share most of their shape.

## 6. Review the Model: Layers, Inputs and Outputs

Before you cut into the network, look at how it is put together. `model.inputs` and `model.outputs` show the shapes that go in and come out, where `None` stands for the number of images in a batch, which is not fixed. `model.layers` lists the layers in order, and every layer has its own `output` with a shape.

In [ ]:
print("Input: ", model.inputs[0].shape)
print("Output:", model.outputs[0].shape)
print("Layers:", len(model.layers))
for layer in model.layers[:4] + model.layers[-3:]:
    print(f"{layer.name:28s} {str(layer.output.shape):22s} {layer.count_params():>10,} parameters")

## 7. What the Layers Respond to at Different Depths

In the convolutional networks notebooks you saw that early layers detect edges and later layers combine them into larger structures. You can check this in MobileNetV2 by cutting the network after an early and after a late layer, as you did with LeNet-5: a [`keras.Model`](https://keras.io/api/models/model/) with the inputs of the network and the output of one of its layers. Building a model by connecting the outputs of layers in this way is called the **functional API**.

Complete the line that creates the cut model in `plot_maps`, then run the cell. The layer names come from `model.summary()`: `block_1_expand_relu` is near the input and outputs 112 × 112 × 96 feature maps, `block_13_expand_relu` is near the end and outputs 14 × 14 × 576.

In [ ]:
def plot_maps(layer_name):
    """Plot the photo and the first six feature maps of one layer of MobileNetV2."""
    cut = keras.Model(inputs=model.inputs, outputs=model.get_layer(layer_name).output)
    maps = cut.predict(prepared, verbose=0)[0]
    fig, axes = plt.subplots(1, 7, figsize=(16, 2.6))
    axes[0].imshow(photo)
    axes[0].set_title("photo")
    for i, ax in enumerate(axes[1:]):
        ax.imshow(maps[:, :, i], cmap="gray")
        ax.set_title(f"map {i}")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"{layer_name}: {maps.shape[0]} x {maps.shape[1]} x {maps.shape[2]}")
    plt.show()


plot_maps("block_1_expand_relu")
plot_maps("block_13_expand_relu")

The maps of `block_1_expand_relu` still have a resolution of 112 × 112 and look like filtered versions of the photo: some respond to the outline of the llama and the fence posts, others to the texture of its wool or to fine vertical structures. The maps of `block_13_expand_relu` have only 14 × 14 values, and each value summarises a large area of the photo. Most of them are dark, and only a few regions respond strongly, for example the lower part of the body in map 3. They no longer show edges but the presence of larger structures, and together the 576 maps describe the image well enough for the classifier to recognise a llama.

## 8. Where a Pretrained Network Reaches Its Limits

A pretrained network can only answer with one of the classes it was trained on. The photo below shows a bean leaf with **angular leaf spot**, a plant disease. ImageNet has no class for bean leaves or their diseases, so what does MobileNetV2 answer?

In [ ]:
for label, probability in classify(model, mobilenet_v2, PHOTOS / "bean_leaf.jpg"):
    print(f"{label:20s} {probability:.3f}")
plt.imshow(keras.utils.load_img(PHOTOS / "bean_leaf.jpg"))
plt.axis("off")
plt.show()

MobileNetV2 answers sea urchin (0.125), trilobite (0.074) and cicada (0.031). All three probabilities are low, so the network is not confident in any of its 1,000 classes, and none of them describes a bean leaf. The answers are probably not random either: the spotted surface of the leaf resembles the texture of a sea urchin. The network's features capture texture and shape well; only its classifier lacks the right classes. That is exactly the situation in which transfer learning helps.

## Try It: A Second Pretrained Network

Classify the five example photos with [EfficientNetB0](https://keras.io/api/applications/efficientnet/) (2019, 5.3 million parameters, a 21 MB download) and compare its top-3 classes and probabilities with those of MobileNetV2. Which photo does EfficientNetB0 classify with a clearly higher probability than MobileNetV2? Its functions live in the module `keras.applications.efficientnet`. Its `preprocess_input` leaves the pixels unchanged, because the scaling is part of the network itself.

In [ ]:
from keras.applications import efficientnet

efficient_model = efficientnet.EfficientNetB0(weights="imagenet")
print(f"EfficientNetB0: {efficient_model.count_params():,} parameters, MobileNetV2: {model.count_params():,}")

for name in photos:
    top3 = classify(efficient_model, efficientnet, PHOTOS / name)
    print(f"{name:20s}", ", ".join(f"{label} {p:.2f}" for label, p in top3))

**Result.** EfficientNetB0 also recognises all five photos, with the laptop again classified as "notebook" (0.68). It is more confident than MobileNetV2 on the electric guitar (0.78 against 0.44) and similar on the others (grand piano 0.93, umbrella 0.89, llama 0.82). It has 5.3 million parameters, 1.5 times as many as MobileNetV2's 3.5 million, and is more accurate on ImageNet; MobileNetV2 is the faster choice when computing power is limited.

## Summary

- Networks trained on ImageNet recognise 1,000 classes of objects. `keras.applications` provides them with their weights, which are downloaded once and cached.
- A photo has to be prepared exactly as the network was trained: resized, given a batch axis and passed through the network's own `preprocess_input`.
- In VGG16 three quarters of the 138 million parameters sit in the first dense layer; the convolutional base is small in comparison.
- MobileNetV2 replaces standard convolutions with depthwise separable ones, which need about 8 times fewer weights, and uses a single dense layer after global average pooling: 3.5 million parameters in total.
- Early layers respond to edges and colours, late layers to larger structures of the object.
- A pretrained network can only choose among its own classes. For a new task, such as bean diseases, you reuse its convolutional layers and train a new classifier: transfer learning, the topic of notebook 02.

Next, continue with [02 - Transfer Learning](02_transfer_learning.ipynb).

## References

- Simonyan, K. and Zisserman, A. (2015). [Very Deep Convolutional Networks for Large-Scale Image Recognition](https://arxiv.org/abs/1409.1556): the VGG paper.
- Sandler, M., Howard, A., Zhu, M., Zhmoginov, A. and Chen, L.-C. (2018). [MobileNetV2: Inverted Residuals and Linear Bottlenecks](https://arxiv.org/abs/1801.04381).
- Bendersky, E. (2018). [Depthwise separable convolutions for machine learning](https://eli.thegreenplace.net/2018/depthwise-separable-convolutions-for-machine-learning/): an illustrated explanation of the depthwise and the pointwise step.
- Howard, A. et al. (2017). [MobileNets: Efficient Convolutional Neural Networks for Mobile Vision Applications](https://arxiv.org/abs/1704.04861): the paper that introduced depthwise separable convolutions for mobile networks.
- Russakovsky, O. et al. (2015). [ImageNet Large Scale Visual Recognition Challenge](https://arxiv.org/abs/1409.0575): the dataset and the competition.
- Keras API: [Keras Applications](https://keras.io/api/applications/), with the size and accuracy of every pretrained network.
- Fei-Fei, L., Fergus, R. and Perona, P. (2004). [Caltech-101](https://data.caltech.edu/records/mzrjq-6wc02), CC BY 4.0: the source of the example photos.
- Makerere AI Lab (2020). [Bean disease dataset](https://github.com/AI-Lab-Makerere/ibean), MIT licence: the source of the bean leaf photo.
- The structure follows the pretrained networks notebooks of the Spiced Academy deep learning course.